# F6 · Bernoulli elite search over a discrete latent: calibrating a frozen quantum generator
**Series F: Quantum Meets Machine Learning, Post 6** &nbsp;|&nbsp; companion *primer* notebook

Yu and Batista (manuscript, 2026) take a pretrained **quantum response network** from their Qydra virtual-cell model, freeze it, replace its ten-dimensional latent by a **ten-bit binary code** $b\in\{0,1\}^{10}$ ($z_i=2b_i-1$), and calibrate the code directly against the population-level **six-component Virtual Cell Challenge (VCC) score**. The code is drawn from a factorized Bernoulli distribution that is refit to the elite samples at every cycle (a discrete **cross-entropy method**, CEM), and, because there are only $2^{10}=1024$ codes, the search is judged against **exhaustive enumeration**. On a synthetic 24-bit problem the optimizer finds the exact optimum within four cycles; on 30 held-out H1 perturbations the score goes from $+0.1914$ (default latent) to $+0.2233$, against $+0.2288$ for the best enumerated code; and four actors that try to predict the code from ESM-2 gene features do not beat the default.

The repository `SamuelYueYu/Qydra` holds the model, the data adapters and the scoring code; **the Bernoulli-search scripts are not in it** (the manuscript says that the archival release will add them). This primer therefore implements the search from the manuscript's equations and checks what can be checked without the model: the **scoring arithmetic** (the score is the mean of six reference-scaled components, and the README's numbers obey it); the **update rule** and its pseudocount ceiling (the saturation values $H=0.0443$ and $P=0.9903$ of the manuscript's Table 1 follow from $K=102$ elites); a **reproduction of Table 1's trajectory** on a synthetic problem; the **evaluation budget** against enumeration on noisy ten-bit landscapes; the **winner's curse** of in-sample selection; how unusual **26 distinct codes among 30 genes** is; and a small **amortization** experiment. Part II runs the real pieces: the repository's simulator, its scorer and its held-out list.

> **Two parts.** **Part I** needs only NumPy, SciPy, pandas and matplotlib and takes about two minutes on a CPU. **Part II** needs PyTorch and the repository (it clones it; about 200 MB); each stage is guarded.
>
> **Reading the code.** Every section starts with a short **Stop and predict** question: answer it before running the cell.

| Section | What you do |
|---|---|
| 1 | the six-component score and its reference scaling |
| 2 | the Bernoulli/CEM update, step sizes and the pseudocount ceiling |
| 3 | synthetic 24-bit recovery against Table 1 |
| 4 | the evaluation budget against enumeration, with noise |
| 5 | in-sample selection and the winner's curse |
| 6 | how many different codes do 30 genes need? |
| 7 | amortization: can a linear actor predict the code? |
| **Part II** | scipy, the repository's simulator, scorer and held-out list |

## 0. Setup

In [ ]:
import math, time, itertools, importlib.util, os, sys, re, json, subprocess
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
np.set_printoptions(precision=4, suppress=True, linewidth=120)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
COLORS = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")

## 1. The score: six components, reference-scaled, equal weight
The reward of the search is the VCC score. In the repository's implementation of the scorer (`src/metrics_vcc2026.py`) each of six metrics $u$ (perturbation discrimination, expression error, differential-expression fold-change error, direction fidelity, direction reach, significance overlap) is **scaled between two anchors measured on the reference data**: the *baseline* $b$ (the context's mean perturbation response, scored as an ordinary submission; $0$ on the scale) and the *replicate* $r$ (the reference split into halves and scored against itself; $1$ on the scale), $s=(u-b)/(r-b)$. The expression error is clamped to $[0,1]$, the fold-change error is floored at $-6$, and the other four are unclamped; the context's score is the **equal-weight mean of the six**, which is the manuscript's Eq. 6. The README reports the six scaled components for four entries of the validation board.
**Stop and predict:** is each README "overall" the mean of the six components printed beside it?

In [ ]:
def scale(u, b, r, name=""):
    s = (u - b) / (r - b)
    if name == "mse": s = float(np.clip(s, 0, 1))
    if name == "nmae": s = max(s, -6.0)
    return s
print("scaling examples: u at the baseline ->", scale(0.40, 0.40, 0.90), "| at the replicate ->", scale(0.90, 0.40, 0.90), "| above the replicate (unclamped, pds) ->", round(scale(1.20, 0.40, 0.90), 2), "| mse below baseline, clamped ->", scale(0.30, 0.40, 0.90, "mse"), "| nmae far below, floored ->", scale(-5, 0.4, 0.9, "nmae"))
README = {"015 (B1+GQ+B2+ARA-B)": ([0.7341, 0.1364, 0.1566, -0.0138, 0.1288, 0.0031], 0.19088), "SPB": ([0.7335, 0.1579, 0.1559, -0.0139, 0.1274, 0.0031], 0.19396), "MBLB30+SPB": ([0.7408, 0.1955, 0.1276, -0.0111, 0.1323, 0.0037], 0.19813), "MBLBm15+SPB (FINAL's recipe)": ([0.7350, 0.1908, 0.1528, -0.0123, 0.1362, 0.0033], 0.20097)}
rows = [(k, round(float(np.mean(v)), 5), o, round(float(np.mean(v)) - o, 5)) for k, (v, o) in README.items()]
print(pd.DataFrame(rows, columns=["entry", "mean of the six components", "printed overall", "difference"]).to_string(index=False)); print("the six components are pds, mse, nmae, fid, reach, jac; one component (pds, about +0.73) carries most of the sum: without it the mean of the other five is", round(float(np.mean(README["015 (B1+GQ+B2+ARA-B)"][0][1:])), 4))
gain = (0.2233 - 0.1914) / (0.2288 - 0.1914); print(f"the manuscript's held-out H1 numbers: default 0.1914, Bernoulli 0.2233, best enumerated 0.2288: the search recovers {gain:.1%} of the available gain ({0.2233 - 0.1914:.4f} of {0.2288 - 0.1914:.4f}); the remaining {0.2288 - 0.2233:.4f} is the gap to enumeration")
assert max(abs(r_[3]) for r_ in rows) < 5e-5 and abs(gain - 0.853) < 0.002

## 2. The Bernoulli/CEM update
Eqs. 3--10 of the manuscript: a code $b$ is decoded as $z=2b-1$; the policy is $\pi_\theta(b)=\prod_ip_i^{b_i}(1-p_i)^{1-b_i}$ with $p_i=\sigma(\theta_i)$ and $\theta=0$ at the start; $N$ codes are sampled and scored; the top $K$ form the elite set; with $c_i$ the number of elite samples with $b_i=1$, $q_i=(c_i+\tfrac12)/(K+1)$, $\lambda_i=\mathrm{logit}\,q_i$ and
$$\theta_i^{(t+1)}=(1-\alpha_t)\theta_i^{(t)}+\alpha_t\lambda_i,\qquad \alpha_t=\frac{0.40}{(1+0.015\,t)^{0.20}}.$$
The manuscript does not state $N$ and $K$ for the runs; the workflow figure (Figure 3) gives $N=1{,}024$ codes per cycle, the top $10\%$ ($K=102$) as the elite and $120$ cycles. Diagnostics: the mean binary entropy $H$ (Eq. 14) and the polarization $P$ (Eq. 15).
**Stop and predict:** with $K=102$, how close to a certain bit can a probability ever get, and what do $H$ and $P$ equal there?

In [ ]:
sigm = lambda x: 1 / (1 + np.exp(-x)); logit = lambda p: np.log(p / (1 - p))
alpha = lambda t: 0.40 / (1 + 0.015 * t) ** 0.20
def Hbin(p): p = np.clip(np.asarray(p, float), 1e-12, 1 - 1e-12); return float(np.mean(-(p * np.log2(p) + (1 - p) * np.log2(1 - p))))
def Ppol(p): return float(np.mean(2 * np.abs(np.asarray(p, float) - 0.5)))
def cem(reward, d, N=1024, K=102, T=80, seed=0, fresh=1024, opt=None):
    """sample N codes, score, keep the top K, pseudocount frequencies, smooth in logit space; fresh samples are drawn after each update for the diagnostics only"""
    rg = np.random.default_rng(seed); th = np.zeros(d); hist = []; seen = set()
    for t in range(T):
        p = sigm(th); B = (rg.random((N, d)) < p).astype(int); r = reward(B, rg); seen.update(map(tuple, B))
        el = np.argsort(-r, kind="stable")[:K]; q = (B[el].sum(0) + 0.5) / (K + 1); a = alpha(t); th = (1 - a) * th + a * logit(q)
        p = sigm(th); Fr = (rg.random((fresh, d)) < p).astype(int); rf = reward(Fr, rg)
        hist.append(dict(cycle=t, mean=rf.mean(), best=rf.max(), H=Hbin(p), P=Ppol(p), acc=float(np.mean((p > 0.5) == (opt == 1))) if opt is not None else np.nan, uniq=len(seen)))
    return pd.DataFrame(hist), th
K = 102; pmax = (K + 0.5) / (K + 1)
print(f"largest probability the update can reach with K = {K}: q = (K + 1/2)/(K + 1) = {pmax:.5f}; logit = {logit(pmax):.3f}; there H = {Hbin([pmax]):.4f} and P = {Ppol([pmax]):.4f}; a bit is wrong with probability {1 - pmax:.5f}, so all 24 bits of a sample are right with probability {pmax ** 24:.3f}")
print("step size: " + ", ".join(f"alpha_{t} = {alpha(t):.3f}" for t in (0, 4, 20, 79, 119)) + "; the logit of a bit moves a fraction alpha of the way to the target at each cycle")
assert abs(Hbin([pmax]) - 0.0443) < 5e-5 and abs(Ppol([pmax]) - 0.9903) < 5e-5

## 3. Synthetic 24-bit recovery against Table 1
The manuscript's synthetic test has a hidden 24-bit optimum and reports, after each update, the mean and best reward of fresh Bernoulli samples, $H$, $P$ and the modal bit accuracy (its Table 1). The reward function is not given; we use the simplest one that decreases with the Hamming distance $d_H$ from the optimum, $R=1-d_H/24$, and compare the trajectories.
**Stop and predict:** which columns of Table 1 will our run reproduce, and which will differ because we do not know the reward?

In [ ]:
d = 24; opt = np.random.default_rng(1).integers(0, 2, d); rew = lambda B, rg: 1 - np.abs(B - opt).sum(1) / d
H1, th = cem(rew, d, opt=opt, seed=0); T1 = pd.DataFrame({"cycle": [0, 4, 7, 10, 20, 79], "mean": [0.5284, 0.8448, 0.9816, 0.9923, 0.9942, 0.9940], "best": [0.8620, 0.9993, 0.9993, 0.9993, 0.9993, 0.9993], "H": [0.9826, 0.5892, 0.1202, 0.0549, 0.0444, 0.0443], "P": [0.1336, 0.6657, 0.9651, 0.9875, 0.9903, 0.9903], "acc": [0.9583, 1.0, 1.0, 1.0, 1.0, 1.0]}).set_index("cycle")
ours = H1.set_index("cycle").loc[T1.index, ["mean", "best", "H", "P", "acc"]]; cmp_ = pd.concat({"manuscript": T1, "this run (R = 1 - d_H/24)": ours}, axis=1).round(4); print(cmp_.to_string())
print("\nthe entropy and polarization trajectories (H, P) agree with the manuscript's to within", round(float(np.abs(ours[["H", "P"]].values[1:] - T1[["H", "P"]].values[1:]).max()), 3), "from cycle 4 on, and reach its saturation values 0.0443 and 0.9903 exactly; the mean reward at cycle 0 is higher here (0.57 against 0.53): the manuscript's reward falls faster at large distance than a Hamming fraction does")
first_exact = int(H1.cycle[H1.acc >= 1.0].iloc[0]); print(f"the modal code equals the hidden optimum from cycle {first_exact} (the manuscript: by cycle 4); fresh samples are exactly optimal with probability about {pmax ** 24:.2f} at saturation, so the mean fresh reward saturates at {H1['mean'].iloc[-1]:.4f} here (the manuscript's {T1['mean'].iloc[-1]:.4f} is {T1['mean'].iloc[-1] / T1['best'].iloc[-1]:.4f} of its best reward {T1['best'].iloc[-1]}; ours {H1['mean'].iloc[-1] / H1['best'].iloc[-1]:.4f})")
cost = 80 * 1024; print(f"evaluations used by the 80 cycles: {cost:,} of the 2^24 = {2 ** 24:,} codes ({cost / 2 ** 24:.2%}); the modal code was exact after {(first_exact + 1) * 1024:,} evaluations")
assert np.abs(ours[["H", "P"]].values[1:] - T1[["H", "P"]].values[1:]).max() < 0.04 and first_exact <= 4 and abs(ours.H.iloc[-1] - 0.0443) < 1e-3

## 4. The evaluation budget against enumeration, with noise
The manuscript calls the stochastic search successful when it approaches the enumerated optimum "while requiring substantially fewer scored candidates than full enumeration". Enumeration scores $1{,}024$ codes. A cycle with $N=1{,}024$ samples draws as many codes as there are in the whole space (with repeats), and the search takes about $20$ cycles to converge. Two things decide whether that is a saving: how many **distinct** codes the search actually scores, and whether the reward is **noisy**. Here the final cells come from sampled and rounded counts ("stochastic rounding" in the submission recipe), so one evaluation of a code is a random draw, and the argmax of $1{,}024$ noisy evaluations is itself biased upward (Section 5).
We compare, on ten-bit landscapes with a known true reward (unit variance over the codes) and Gaussian evaluation noise: enumeration (one evaluation of every code, take the argmax); CEM with $N=1024,K=102$; $N=256,K=26$; $N=64,K=6$ (each for $20$ cycles, caching nothing, final modal code); landscapes: additive in the bits, pairwise (Ising-like), and unstructured (independent per code). The measure is the **gap**, in standard deviations of the true reward, between the best code and the chosen one.
**Stop and predict:** with $N=1{,}024$, what fraction of the $1{,}024$ codes does the search score within $20$ cycles?

In [ ]:
d10 = 10; codes10 = ((np.arange(1 << d10)[:, None] >> np.arange(d10)) & 1) * 2 - 1; idx10 = lambda B: (B * (1 << np.arange(d10))).sum(1)
def landscape(kind, rg):
    if kind == "additive": true = codes10 @ rg.normal(size=d10)
    elif kind == "pairwise": true = codes10 @ (0.6 * rg.normal(size=d10)) + np.einsum("ni,ij,nj->n", codes10, np.triu(0.5 * rg.normal(size=(d10, d10)), 1), codes10)
    else: true = rg.normal(size=1 << d10)
    return (true - true.mean()) / true.std()
def compare(kind, sigma, reps=20):
    gaps = {k: [] for k in ("enumeration", "CEM N=1024", "CEM N=256", "CEM N=64")}; uniq = {k: [] for k in ("CEM N=1024", "CEM N=256", "CEM N=64")}
    for r in range(reps):
        rg = np.random.default_rng(1000 + r); true = landscape(kind, rg); best = true.max(); noisy = lambda B, g: true[idx10(B)] + sigma * g.normal(size=len(B))
        gaps["enumeration"].append(best - true[np.argmax(true + sigma * rg.normal(size=1 << d10))])
        for name, (N, K_) in {"CEM N=1024": (1024, 102), "CEM N=256": (256, 26), "CEM N=64": (64, 6)}.items():
            h, th_ = cem(noisy, d10, N=N, K=K_, T=20, seed=r, fresh=8); code_ = (sigm(th_) > 0.5).astype(int)[None]; gaps[name].append(best - true[idx10(code_)[0]]); uniq[name].append(h.uniq.iloc[-1])
    return {k: float(np.mean(v)) for k, v in gaps.items()}, {k: float(np.mean(v)) for k, v in uniq.items()}
t0 = time.perf_counter(); rows = []
for kind in ("additive", "pairwise", "unstructured"):
    for sg in (0.0, 1.0):
        g_, u_ = compare(kind, sg); rows.append((kind, sg, *[round(g_[k], 2) for k in g_], *[round(u_[k]) for k in u_]))
BUD = pd.DataFrame(rows, columns=["landscape", "noise sd", "gap: enumeration", "gap: CEM 1024", "gap: CEM 256", "gap: CEM 64", "codes scored: 1024", "codes scored: 256", "codes scored: 64"]); print(BUD.to_string(index=False)); print(f"[{time.perf_counter() - t0:.0f} s]")
print("reading: (i) with N = 1024 the search scores 840 to 1024 of the 1024 codes in 20 cycles, i.e. about the cost of enumeration; N = 64 scores 250 to 375 on the additive and pairwise landscapes; (ii) with evaluation noise the argmax of a single pass over all codes has a gap of about 0.7 to 0.8 sd, and the elite averaging of the CEM beats it on smooth landscapes; (iii) on an unstructured landscape no bit-wise search works (gap about 1 sd) and enumeration is the only reliable reference")
a_ = BUD[BUD.landscape == "additive"]; assert (a_["codes scored: 1024"] > 700).all() and (a_["codes scored: 64"] < 450).all() and BUD[(BUD.landscape == "additive") & (BUD["noise sd"] == 1.0)]["gap: enumeration"].iloc[0] > BUD[(BUD.landscape == "additive") & (BUD["noise sd"] == 1.0)]["gap: CEM 256"].iloc[0]

## 5. In-sample selection and the winner's curse
In the held-out experiment the search is allowed to use the measured cells of the very perturbation it optimizes, and the reported scores (default $+0.1914$, Bernoulli $+0.2233$, best enumerated $+0.2288$) are evaluated on those cells. That is the right design for asking whether better latent modes **exist**, and the manuscript says so. It also means that the best enumerated code is the maximum of $1{,}024$ evaluations, each with its own noise. If an evaluation of code $c$ gives $g_c+\varepsilon_c$ (a true effect plus evaluation noise), the expected apparent gain of the best code over the default is $E[\max_c(g_c+\varepsilon_c)]$, while its true gain is $E[g_{c^\star}]$; for Gaussian effects of sd $\tau$ and noise of sd $\sigma$ the ratio is $\tau^2/(\tau^2+\sigma^2)$. The manuscript does not report the evaluation noise; we show what the reported gain would mean for assumed values, with the gain per gene scaled so that the apparent total is the reported $0.2288-0.1914=0.0374$.
**Stop and predict:** if every apparent difference between codes were noise, what evaluation noise per gene would produce the reported gain of $0.0374$?

In [ ]:
rg = np.random.default_rng(0); A_target = 0.2288 - 0.1914; M, G = 1024, 30
emax = float(np.mean([rg.normal(size=M).max() for _ in range(4000)])); print(f"expected maximum of {M} standard normals: {emax:.3f}; if all variation were noise, the gain 0.0374 would need sd = {A_target / emax:.4f} per code and gene")
rows = []
for share in (0.0, 0.25, 0.5, 0.75, 1.0):                       # share of the code-to-code variance that is evaluation noise
    tot = A_target / emax; sig_n, tau = tot * math.sqrt(share), tot * math.sqrt(1 - share)
    app, tru = [], []
    for _ in range(300):
        g = tau * rg.normal(size=(G, M)); e = sig_n * rg.normal(size=(G, M)); s = np.argmax(g + e, axis=1); app.append((g + e)[np.arange(G), s].mean()); tru.append(g[np.arange(G), s].mean())
    rows.append((share, round(sig_n, 4), round(float(np.mean(app)), 4), round(float(np.mean(tru)), 4), round(float(np.mean(tru) / np.mean(app)), 2)))
WC = pd.DataFrame(rows, columns=["noise share of the variance", "noise sd", "apparent gain", "true gain of the chosen code", "true / apparent"]); print(WC.to_string(index=False))
print("the manuscript's fresh-sample diagnostics concern the Bernoulli samples, not the measured cells; a re-evaluation of the chosen codes with new random seeds for the rounding and the cell draw (or on held-out cells of the same perturbation) would give the noise share directly")
print("the search itself is less exposed than enumeration: it selects on averages of many evaluations (Section 4), and 11 of 30 genes were within 0.001 of the enumerated optimum")
assert abs(WC["apparent gain"].iloc[0] - A_target) < 0.002 and WC["true gain of the chosen code"].iloc[-1] < 0.001 and WC["true gain of the chosen code"].iloc[0] > 0.035

## 6. How many different codes do 30 genes need?
The manuscript reports that the $30$ held-out genes "required 26 different optimized codes", read as evidence that the state-specific solutions are heterogeneous. If each gene's best code were an independent uniform draw from the $1{,}024$ codes, how many distinct codes would $30$ genes have, and how unusual is $26$?
**Stop and predict:** is $26$ more or fewer distinct codes than chance would give?

In [ ]:
rg = np.random.default_rng(0); exp_distinct = 1024 * (1 - (1 - 1 / 1024) ** 30); nd = np.array([len(set(rg.integers(0, 1024, 30))) for _ in range(50000)])
print(f"30 independent uniform draws from 1024 codes: expected distinct {exp_distinct:.2f}; P(distinct <= 26) = {np.mean(nd <= 26):.4f}; P(distinct <= 29) = {np.mean(nd <= 29):.3f}")
rows = []
for Mc in (1024, 512, 256, 128, 64):
    nd2 = np.array([len(set(rg.integers(0, Mc, 30))) for _ in range(20000)]); rows.append((Mc, round(float(nd2.mean()), 2), round(float(np.mean(nd2 <= 26)), 3)))
print(pd.DataFrame(rows, columns=["codes drawn uniformly from", "mean distinct among 30", "P(<= 26)"]).to_string(index=False))
print("26 distinct codes is fewer than 30 independent uniform codes would give (p = 0.0004): the best codes share structure (a few codes are chosen more than once); the number is consistent with an effective repertoire of about 100 codes, so 'highly heterogeneous' is true at the level of the individual code and not at the level of the repertoire")
print("caveat: the stochastic search itself favours some codes (its elite averaging has bit-level biases), so a repeated code is not by itself evidence of a shared biological response")
assert abs(exp_distinct - 29.58) < 0.01 and np.mean(nd <= 26) < 0.002

## 7. Amortization: can a linear actor predict the code?
The actor $p(s)=\sigma(f_\omega(s))$ is trained by the cross-entropy of Eq. 12 against the elite targets $q(s)$ of the searched states ($268$ states in Figure 3: $120$ H1 and $148$ K562 genes) from $5{,}120$-dimensional ESM-2 gene features, and used at inference through $b^\star_i=\mathbb 1[p_i>1/2]$. The reported zero-shot scores ($-0.1773$ to $+0.1889$) do not beat the default. We build the same pipeline on synthetic data to see what is needed for it to work: $268$ training states and $30$ test states with $p=200$ features; each bit of the best code is generated from a few features with strength $s$ plus noise, $s=0$ meaning no relation; a linear (logistic) actor per bit with an $L_2$ penalty.
**Stop and predict:** with no relation between features and codes, what is the held-out bit accuracy and the chance of reproducing a whole code?

In [ ]:
def fit_logreg(X, y, lam, it=300):
    w = np.zeros(X.shape[1]); b = 0.0
    for _ in range(it): p = sigm(X @ w + b); w -= 0.5 * (X.T @ (p - y) / len(y) + lam * w); b -= 0.5 * (p - y).mean()
    return w, b
def amortize(s, p=200, ntr=268, nte=30, d=10, lam=0.05, seed=0):
    rg = np.random.default_rng(seed); X = rg.normal(size=(ntr + nte, p)); W = np.zeros((p, d)); W[:5] = rg.normal(size=(5, d)) * s; codes = (X @ W + rg.normal(size=(ntr + nte, d)) > 0).astype(int); P = np.zeros((nte, d))
    for j in range(d): w, b = fit_logreg(X[:ntr], codes[:ntr, j], lam); P[:, j] = sigm(X[ntr:] @ w + b)
    pred = (P > 0.5).astype(int); return (pred == codes[ntr:]).mean(), np.mean((pred == codes[ntr:]).all(1))
t0 = time.perf_counter(); rows = []
for s in (0.0, 0.5, 1.0, 2.0, 4.0): r_ = np.array([amortize(s, seed=k) for k in range(20)]); rows.append((s, r_[:, 0].mean().round(3), r_[:, 1].mean().round(3)))
AMZ = pd.DataFrame(rows, columns=["relation strength s", "held-out bit accuracy", "held-out exact code"]); print(AMZ.to_string(index=False)); print(f"chance: bit accuracy 0.500, exact code {1 / 1024:.4f} [{time.perf_counter() - t0:.0f} s]")
print("so even when the codes are generated from five features of two hundred, a linear actor gets about three bits in four right at best and the exact code in about one case in ten; with no relation it is at chance: a null result of the actor is what an absent or weak relation gives, and it cannot say which")
print("the real problem is harder: 5,120 features for 268 states, the codes are the noisy output of a stochastic search, and the score landscape varies with the cell context (the manuscript's own explanation)")
fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.7)); ax[0].plot(H1.cycle, H1.H, color=COLORS["blue"], label="H (this run)"); ax[0].plot(H1.cycle, H1.P, color=COLORS["orange"], label="P (this run)"); ax[0].scatter(T1.index, T1.H, color=COLORS["blue"], marker="s", s=22, label="H (Table 1)"); ax[0].scatter(T1.index, T1.P, color=COLORS["orange"], marker="s", s=22, label="P (Table 1)"); ax[0].axhline(0.0443, color="gray", ls=":", lw=0.8); ax[0].set(xlabel="cycle", title="(a) Bernoulli/CEM on 24 bits, N = 1024, K = 102"); ax[0].legend(frameon=False, fontsize=7)
ax[1].plot(AMZ["relation strength s"], AMZ["held-out bit accuracy"], "o-", color=COLORS["green"]); ax[1].axhline(0.5, color="gray", ls=":", lw=0.8); ax[1].set(xlabel="relation strength between features and code bits", ylabel="held-out bit accuracy", title="(b) a linear actor on 268 training states", ylim=(0.45, 0.85)); plt.tight_layout(); plt.show()
assert AMZ["held-out bit accuracy"].iloc[0] < 0.53 and AMZ["held-out bit accuracy"].iloc[-1] > 0.7

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.7)); add = BUD[BUD.landscape == "additive"]
x = np.arange(2); ax[0].bar(x - 0.2, add["codes scored: 1024"], 0.2, color=COLORS["red"], label="N = 1024"); ax[0].bar(x, add["codes scored: 256"], 0.2, color=COLORS["orange"], label="N = 256"); ax[0].bar(x + 0.2, add["codes scored: 64"], 0.2, color=COLORS["green"], label="N = 64"); ax[0].axhline(1024, color="k", ls="--", lw=0.8, label="enumeration")
ax[0].set(xticks=x, xticklabels=["no noise", "noise sd 1"], ylabel="distinct codes scored in 20 cycles", title="(a) cost of the search (additive landscape)"); ax[0].legend(frameon=False, fontsize=7, loc="lower right")
ax[1].plot(WC["noise share of the variance"], WC["apparent gain"], "o-", color=COLORS["blue"], label="apparent gain of the best code"); ax[1].plot(WC["noise share of the variance"], WC["true gain of the chosen code"], "s-", color=COLORS["red"], label="true gain of that code"); ax[1].set(xlabel="share of code-to-code variance that is evaluation noise", ylabel="gain over the default code", title="(b) winner's curse of choosing among 1024 codes"); ax[1].legend(frameon=False, fontsize=7)
plt.tight_layout(); plt.show()

## 8. Your turn
**Exercise 1.** Run Section 3 with $K=26$ ($10\%$ of $256$) and with $K=10$. What are the saturation values of $H$ and $P$, and what does the probability of an exactly optimal fresh sample become?
**Exercise 2.** Replace the Hamming reward by $\exp(-d_H/3)$ and by $\exp(-(d_H/3)^2)$ and see whether the cycle-0 mean reward can be brought to $0.528$ while keeping the $H,P$ trajectory.
**Exercise 3.** In Section 4 give the landscape a few strong pairwise couplings and find the $N$ at which the CEM stops matching enumeration.
**Exercise 4.** In Section 5 use $\sigma$ estimated from two repeated evaluations of the same code to place the Qydra result on the table.
**Exercise 5.** In Section 7 replace the linear actor by one that also receives a cell-context vector, and let the code depend on both.

## 9. Self-check

In [ ]:
assert abs(Hbin([pmax]) - 0.0443) < 5e-5 and abs(gain - 0.853) < 0.002 and first_exact <= 4 and np.mean(nd <= 26) < 0.002
print("All checks passed: you reproduced the numbers in the post.")

# Part II. scipy, and the repository's simulator, scorer and held-out list
> **Read this first.** Part II has three guarded stages. **(1)** scipy: our logit, sigmoid and binary entropy against `scipy.special` and `scipy.stats.entropy`. **(2)** The repository `SamuelYueYu/Qydra`: the cell clones it (about 200 MB, shallow) and checks (a) that the held-out list `analysis/holdout_seed0_ho30.json` has the $30$ genes of the manuscript, (b) that no file of the repository contains a Bernoulli-search or CEM implementation (the manuscript says that the scripts come with the archival release), (c) the **scorer's scaling**: `metrics_vcc2026.scale_scores` against our $s=(u-b)/(r-b)$ with its clamps and the equal-weight mean, and (d) the **state-vector simulator** `src/qsim.py` (needs PyTorch): a single-qubit RY rotation gives $\langle Z\rangle=\cos\theta$, and an RY followed by a CNOT gives a Bell-type state with probabilities $\cos^2(\theta/2)$ and $\sin^2(\theta/2)$ on the two correlated outcomes. **(3)** The real experiment (the frozen network and the VCC data) is not run here: the model checkpoints and the challenge data are not in the repository; the primer's `cem` and an enumeration loop accept any reward function, so with your own model you pass `reward(B, rg)`, where `B` is an array of codes and the return value is the score of each. Each stage is guarded; the last cell lists the checks that passed.

In [ ]:
checks = []
def check(label, ok): checks.append((label, bool(ok))); print(("   ok   " if ok else "  FAIL  ") + label)
try:
    from scipy.special import expit, logit as slogit; x_ = np.linspace(-6, 6, 25); p_ = np.linspace(0.01, 0.99, 25)
    check("our sigmoid and logit equal scipy.special.expit and logit", np.allclose(sigm(x_), expit(x_)) and np.allclose(logit(p_), slogit(p_)))
    pv = np.array([0.1, 0.5, 0.9, pmax]); check("our mean binary entropy (bits) equals scipy.stats.entropy of the Bernoulli distributions", abs(Hbin(pv) - np.mean([stats.entropy([p, 1 - p], base=2) for p in pv])) < 1e-12)
    th_ = np.array([0.0, 1.0, -2.0]); q_ = np.array([0.7, 0.4, 0.99]); a_ = alpha(5); new_ = (1 - a_) * th_ + a_ * logit(q_); check("one smoothed update of the logits equals the formula theta' = (1 - alpha) theta + alpha logit(q)", np.allclose(new_, th_ + a_ * (logit(q_) - th_)))
except Exception as e_: print("scipy stage could not be completed:", type(e_).__name__, e_)

In [ ]:
REPO = "Qydra"
try:
    if not os.path.isdir(REPO): subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/SamuelYueYu/Qydra.git", REPO])
    ho = json.load(open(os.path.join(REPO, "analysis", "holdout_seed0_ho30.json"))); print("held-out genes:", " ".join(ho["holdout"]), "| seed", ho["seed"], "| min_reliability", ho["min_reliability"])
    check("the held-out list has the 30 perturbations of the manuscript's H1 panel", len(ho["holdout"]) == 30 and len(set(ho["holdout"])) == 30)
    hits = []
    for root, _, files in os.walk(REPO):
        if ".git" in root: continue
        for f in files:
            if f.endswith((".py", ".md", ".sh", ".json", ".ipynb")):
                t_ = open(os.path.join(root, f), errors="ignore").read().lower()
                if re.search(r"\b(cem|cross-entropy method|elite)\b|bernoulli.{0,40}(search|policy|latent)", t_): hits.append(os.path.join(root, f))
    print("files that mention a CEM, elite or Bernoulli-search implementation:", hits if hits else "none")
    check("the repository contains no Bernoulli-search implementation (the manuscript: provided with the archival release)", not hits)
    sys.path.insert(0, os.path.join(REPO, "src")); import metrics_vcc2026 as mv
    names = mv.METRIC_NAMES; raw = dict(zip(names, [0.62, 0.90, 0.55, 0.40, 0.33, 0.48])); base = dict(zip(names, [0.50, 0.60, 0.40, 0.20, 0.10, 0.20])); rep = dict(zip(names, [0.80, 0.30, 0.60, 0.50, 0.40, 0.60]))
    sc = mv.scale_scores(raw, base, rep); mine = [scale(raw[n], base[n], rep[n], "mse" if "mse" in n else ("nmae" if "nmae" in n else "")) for n in names]
    print("repository scale_scores:", {k: round(v, 4) for k, v in sc.items()}); print("our scaling:           ", [round(v, 4) for v in mine], "mean", round(float(np.mean(mine)), 4))
    check("scale_scores equals (u - b)/(r - b) with the clamp of the expression error and the floor of the fold-change error", np.allclose([sc[n] for n in names], mine))
    check("the context score is the equal-weight mean of the six scaled components", abs(sc["context_score"] - np.mean(mine)) < 1e-12)
except Exception as e_: print("repository stage could not be completed:", type(e_).__name__, str(e_)[:300])

In [ ]:
try:
    import torch, qsim
    th = 0.7; st = qsim.zero_state(2, 1, "cpu"); st = qsim.ry(st, 2, 0, torch.tensor(th)); ez = qsim.expval_z(st, 2, 2).detach().cpu().numpy().ravel(); print("RY(0.7) on wire 0: <Z> per wire =", ez.round(4), "| cos(0.7) =", round(math.cos(th), 4))
    check("a single RY(theta) gives <Z> = cos(theta) on the rotated wire and +1 on the other (state-vector simulator, qsim.py)", abs(max(ez) - 1.0) < 1e-5 and abs(min(ez) - math.cos(th)) < 1e-5)
    s2 = qsim.cnot(st, 2, 0, 1); pr = qsim.probabilities(s2).detach().cpu().numpy().ravel(); print("probabilities after RY then CNOT:", pr.round(4), "| cos^2(theta/2), sin^2(theta/2) =", round(math.cos(th / 2) ** 2, 4), round(math.sin(th / 2) ** 2, 4))
    check("RY followed by CNOT gives probabilities cos^2(theta/2) and sin^2(theta/2) on two correlated outcomes and a unit total", abs(pr.sum() - 1) < 1e-5 and abs(np.sort(pr)[-1] - max(math.cos(th / 2) ** 2, math.sin(th / 2) ** 2)) < 1e-5 and np.sum(pr > 1e-6) == 2)
except Exception as e_: print("simulator stage could not be completed:", type(e_).__name__, str(e_)[:300])
if checks:
    ok_ = [l for l, p_ in checks if p_]; bad_ = [l for l, p_ in checks if not p_]; print(f"\nPart II: {len(ok_)} of {len(checks)} checks passed")
    for l in bad_: print("  not passed:", l)

### What to look for
* **The held-out list.** It is the seed-0 split of 30 perturbations of the VCC 2025 H1 data; the repository notes that it was pinned so that every feature set scores the same genes.
* **The missing code.** The repository holds the model, its data adapters and the scorer; the search itself is described in the manuscript and in its Supporting Information. The check reports that it is not in the repository.
* **The scorer.** If `scale_scores` agrees with $(u-b)/(r-b)$ and the equal-weight mean, the manuscript's Eq. 6 is the repository's score.
* **The real experiment.** To run the search on the frozen network, supply a reward function that decodes a code with the network, applies the post-network recipe, and scores the 400 predicted cells against the measured cells.